In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Define the parameters based on our internal scenarios
# Baseline: 250 MW continuous load -> 250 MW * 8760 hours = 2,190,000 MWh required annually.
data = {
    'Asset_Scenario': [
        '100% Geothermal (S3)',
        'SMR Nuclear (S5)',
        'Microturbines (S6)',
        'RICE Peaking (S1)',
        'Tiered-Performance BESS'
    ],
    'Nameplate_Capacity_MW': [
        300.0,  # Sized for 250 MW + 50 MW redundancy (N+1)
        278.0,  # Sized tighter (250 MW + ~10% redundancy)
        300.0,  # Sized for 250 MW + redundancy (used in hybrid)
        300.0,  # Sized for peaking and backup
        250.0   # Sized to bridge the full 250 MW load during transients
    ],
    'Projected_Annual_MWh': [
        2190000, # Runs baseload, meets the full 250 MW load for 8760 hours
        2190000, # Runs baseload, meets the full 250 MW load for 8760 hours
        1182600, # Dispatches roughly 45% of the time (hybrid load-following)
        788400,  # Dispatches roughly 30% of the time (peaking/backup)
        164250   # Dispatches short bursts, totaling ~7.5% equivalent annual hours
    ]
}

# 2. Create DataFrame and calculate the metrics
df = pd.DataFrame(data)

# Total Possible MWh = Nameplate MW * 8760 hours
df['Total_Possible_MWh'] = df['Nameplate_Capacity_MW'] * 8760

# Capacity Factor (CF) = (Projected MWh / Total Possible MWh) * 100
df['Capacity_Factor_pct'] = (df['Projected_Annual_MWh'] / df['Total_Possible_MWh']) * 100

# Resilience Reserve (RR) = 100 - CF
df['Resilience_Reserve_pct'] = 100 - df['Capacity_Factor_pct']

# Save to CSV
df.to_csv("asset_utilization_calculations.csv", index=False)

# 3. Generate the Chart using the calculated DataFrame
fig, ax = plt.subplots(figsize=(10, 6))
bar_width = 0.55
index = np.arange(len(df))

# Plot the stacked bars directly from the dataframe calculations
p1 = ax.bar(index, df['Capacity_Factor_pct'], bar_width, label='Operational Capacity Factor (Active Dispatch)', color='#2ca02c', edgecolor='black')
p2 = ax.bar(index, df['Resilience_Reserve_pct'], bar_width, bottom=df['Capacity_Factor_pct'], label='Resilience Reserve (N+1 Standby / Outage Bridging)', color='#98df8a', edgecolor='black')

# Formatting
ax.set_ylabel('Total Physical Nameplate Capacity (%)', fontsize=12, fontweight='bold')
#ax.set_title('Figure 3: Internally Modeled Asset Utilization vs. Resilience Reserve', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(index)
ax.set_xticklabels(df['Asset_Scenario'], rotation=15, ha='right', fontsize=11)
ax.set_ylim(0, 115)

# Horizontal line
ax.axhline(y=100, color='red', linestyle='--', linewidth=1.5, label='Installed Nameplate Capacity ($P_{nameplate}$)')

# Legend and Grid
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

# Save chart
plt.tight_layout()
plt.savefig('internal_asset_utilization.png', dpi=300)
plt.close()